#  AI Resume Screener (Mini HR Tool)

###  Project Goal
This project provides an end-to-end, locally-hosted **Retrieval-Augmented Generation (RAG)** tool designed to simplify HR processes. Instead of manually reviewing hundreds of resumes, recruiters can query this tool using natural language (e.g., *"Who has experience with TensorFlow and AWS?"*) and instantly receive matching candidates along with direct summaries of their relevant skills.

---

###  Project Screenshots

#### 1. Main UI Page
![Main UI Page](content/main_page.png)

#### 2. Candidate Screening - Query Example 1
![Screen Query 1](content/question_1.png)


---

###  How the Pipeline Works

1. **Upload Resumes**: Drop PDF or TXT resumes into the workspace.
2. **Smart Splitting (LangChain)**: Chunks raw files into semantic snippets so no key experience details are lost.
3. **Local Vector Database (Chroma)**: Embeds and indexes the text chunks using standard sentence embeddings.
4. **Local LLM Inference (Qwen-2.5-3B)**: Answers are generated locally on your GPU in half precision (FP16), guaranteeing complete data privacy.
5. **Interactive Interface (Gradio)**: A clear, accessible web portal displaying candidate summaries and original resume sources side-by-side.

In [10]:
# @title
## Why we check GPU availability:
# PyTorch is the underlying tensor engine for Hugging Face models.
# Running LLMs requires a GPU (such as Nvidia Tesla T4 in free Colab) to achieve fast token generation speeds.
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [ ]:
# @title
## Why we load and split documents:
# - Loader: PyMuPDFLoader and TextLoader read PDF/TXT files, maintaining metadata (e.g., candidate name derived from the file name).
# - Splitter: Large files exceed LLM context windows and degrade retrieval quality.
#   RecursiveCharacterTextSplitter splits documents on logical characters (like newlines) into chunks of 800 characters
#   with a 150-character overlap to prevent losing context across chunk boundaries.
import os
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

RESUME_DIR = "resumes"

# Load one resume (PDF or txt) and tag it with the candidate's name
def load_resume(path):
    if path.lower().endswith(".pdf"):
        docs = PyMuPDFLoader(path).load()
    elif path.lower().endswith(".txt"):
        docs = TextLoader(path, encoding="utf-8").load()
    else:
        return []

    candidate = os.path.splitext(os.path.basename(path))[0]   # file name = candidate
    for doc in docs:
        doc.metadata["candidate"] = candidate
    return docs

# Load every resume in the folder
all_docs = []
for filename in os.listdir(RESUME_DIR):
    all_docs.extend(load_resume(os.path.join(RESUME_DIR, filename)))

# Split into overlapping chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(all_docs)

print(f"✅ Loaded {len(all_docs)} pages, split into {len(chunks)} chunks")
print("Candidates:", sorted({c.metadata["candidate"] for c in chunks}))

In [ ]:
# @title
## Why we use Embeddings and ChromaDB:
# - HuggingFaceEmbeddings: Map text chunks into a 384-dimensional vector space using the efficient 'all-MiniLM-L6-v2' model.
#   This captures semantic meaning (e.g., 'ML' is semantically close to 'Machine Learning' or 'TensorFlow').
# - ChromaDB: Stores these embeddings locally. On query, we search this database using cosine/Euclidean distance
#   to retrieve the most relevant resume fragments matching our HR search query.
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Start with a clean database, so re-running this cell never creates duplicates
shutil.rmtree("chroma_resumes", ignore_errors=True)

# 1) Load the embedding model (turns each resume chunk into a vector of numbers)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded")

# 2) Embed every chunk and store it in ChromaDB (saved in the chroma_resumes folder)
vectorstore = Chroma.from_documents(
    documents=chunks,                    # the resume chunks from the splitting step
    embedding=embeddings,
    collection_name="resumes",
    persist_directory="chroma_resumes",
)

# 3) Confirm how many chunks were stored
print(f"✅ Stored {vectorstore._collection.count()} chunks in ChromaDB")

# 4) Quick check: which resumes are closest in meaning to this query?
results = vectorstore.similarity_search("TensorFlow and AWS experience", k=4)
print("\nTop 4 matches:")
for doc in results:
    print(" -", doc.metadata["candidate"])

In [ ]:
# @title
## Why we use this specific LLM:
# We use 'Qwen/Qwen2.5-3B-Instruct' because it is optimized for instruction-following and fits
# within the memory constraints (VRAM) of a free-tier Colab GPU (Tesla T4) when loaded in FP16 (half-precision).
# 'device_map="auto"' ensures model weights are loaded and processed directly on our active GPU.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the LLM (Qwen2.5-3B-Instruct) on the Colab GPU in half precision
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision: fits easily on a T4
    device_map="auto",           # put the model on the GPU automatically
)
print("✅ LLM loaded")

In [ ]:
# @title
## Why we build this pipeline:
# - retriever: Automatically fetches top K relevant chunks from our Chroma VectorDB.
# - build_messages: Formulates a structured system prompt directing the LLM to behave strictly as an HR agent,
#   preventing hallucinations by instructing it to ONLY screen based on the provided candidates' chunks.
# - generate: Formats the text template, tokenizes it, transfers inputs to CUDA, generates predictions, and decodes the result.
# - screen: The orchestrator bridging retrieval and generation.
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})


# Build the chat messages: retrieved resumes + the HR request
def build_messages(query, docs):
    context = "\n\n".join(
        f"[Candidate: {d.metadata['candidate']}]\n{d.page_content}" for d in docs
    )
    system = (
        "You are an HR assistant that screens resumes. "
        "Use ONLY the resumes provided. A candidate matches only if the resume "
        "explicitly mentions EVERY requirement in the request. "
        "Answer with a short list: candidate name, then one line explaining "
        "which skills matched. If nobody matches, say so."
    )
    user = f"Resumes:\n{context}\n\nRequest: {query}"
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]


# Step F: generate the answer with the LLM
def generate(messages, max_new_tokens=400):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]   # keep only the new text
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Full pipeline: retrieve -> prompt -> generate, with status messages
def screen(query):
    docs = retriever.invoke(query)
    print(f"✅ Retrieval done: {len(docs)} resumes found")
    messages = build_messages(query, docs)
    print("✅ Prompt built")
    answer = generate(messages)
    print("✅ Generation done")
    return answer, docs


# Test it
answer, docs = screen("Find candidates with TensorFlow + AWS experience")
print("\nAnswer:\n", answer)
print("\nRetrieved:", [d.metadata["candidate"] for d in docs])
print("\n✅ Resume screener completed successfully")

In [ ]:
# @title
## Why we use Gradio:
# It instantly wraps our backend Python logic ('screen' function) with a visual, shareable GUI.
# Users can query the tool using predefined examples or custom inputs without running raw Python code blocks.
import gradio as gr

# Called by the UI: runs the screener and formats the two output boxes
def screener_ui(query):
    if not query.strip():                         # ignore empty input
        return "Please type a request.", ""

    answer, docs = screen(query)                  # retrieve + generate (Steps E and F)

    # Show which resumes were retrieved, with a short preview of each
    sources = "\n\n".join(
        f"{d.metadata['candidate']}\n{d.page_content[:200]}..." for d in docs
    )
    return answer, sources

demo = gr.Interface(
    fn=screener_ui,
    inputs=gr.Textbox(
        label="HR request",
        placeholder="e.g. Find candidates with TensorFlow + AWS experience",
        lines=2,
    ),
    outputs=[
        gr.Textbox(label="Matching candidates", lines=8),
        gr.Textbox(label="Retrieved resumes", lines=10),
    ],
    title="ဣ‍ဢ AI Resume Screener (Mini HR Tool)",
    description="Ask for skills or experience. Answers come only from the uploaded resumes.",
    examples=[
        ["Find candidates with TensorFlow + AWS experience"],
        ["Who has experience with Kubernetes?"],
        ["Find a frontend developer with React"],
        ["Who knows Java and PostgreSQL?"],
    ],
    flagging_mode="never",                        # hide the Flag button
)

# share=True also gives a public link; debug=True shows errors in the notebook
demo.launch(share=True, debug=True)